In [ ]:
import os 
os.environ['AWS_PROFILE'] = 'admin'
os.environ['HAVEN_DATABASE'] = 'haven'

import plotly.express as px 
import h3 
import numpy as np
import pandas as pd

from mirrorverse.utils import read_data_w_cache

In [ ]:
sql = '''
with risk as (
    select 
        time, 
        epoch,
        h3_index,
        depth_bin,
        max(elevation) as elevation,
        sum(probability) as risk
    from 
        chinook_depth_full_inference_3_1_4 -- chinook_depth_full_inference_3_1_18_2
    group by 
        1, 2, 3, 4
), max_depth_bin as (
    select 
        time, 
        epoch,
        h3_index,
        max(depth_bin) as depth_bin
    from 
        chinook_depth_full_inference_3_1_4 -- chinook_depth_full_inference_3_1_18_2
    group by 
        1, 2, 3
)
select 
    month(time) as month,
    h3_index,
    depth_bin,
    elevation,
    approx_percentile(risk, 0.05) as min_risk_month,
    approx_percentile(risk, 0.95) as max_risk_month
from 
    risk inner join max_depth_bin using (time, epoch, h3_index, depth_bin)
group by 
    1, 2, 3, 4
'''
data = read_data_w_cache(sql)
data['lat'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[0])
data['lon'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[1])
print(data.shape)
data.head()

In [ ]:
data = data[data['depth_bin'] + 100 > -data['elevation']]
print(data.shape)

In [ ]:
from shapely.geometry import Polygon, Point

poly = Polygon(
    [
        (-166, 54.4),
        (-160, 56),
        (-158, 57.2),
        (-153, 62),
        (-149, 62),
        (-146, 62),
        (-140, 60),
        (-136, 58.4),
        (-133, 57.5),
        (-132, 56.0),
        (-131, 55),
        (-125, 50.3),
        (-170, 52.5),
        (-166, 54.4),
    ]
)
data['inside_polygon'] = data.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
fig = px.scatter_mapbox(
    data[(data['month'] == 2)  & (data['depth_bin'] != 25.0)],
    lat='lat',
    lon='lon',
    color='inside_polygon',  # Color points by probability
    size_max=10,  # Adjust as needed
    zoom=3,  # Adjust zoom level
    mapbox_style="carto-positron",  # Choose a map style,
    title='Spatial Pattern of Minimal Probability in Bottom Depth Bin in February',
    range_color=[0.0, 0.3]
)
fig.show()

In [ ]:
data = data[data['inside_polygon']]
data = data[data['lon'] < -145]
data['size'] = 0.01

In [ ]:
fig = px.scatter_mapbox(
    data[(data['month'] == 2)  & (data['depth_bin'] != 25.0)],
    lat='lat',
    lon='lon',
    color='min_risk_month',  # Color points by probability
    size='size',  # Adjust as needed
    size_max=11,  # Adjust as needed
    zoom=4,  # Adjust zoom level
    mapbox_style="carto-positron",  # Choose a map style,
    title='Spatial Pattern of Minimal Probability in Bottom Depth Bin in February',
    #range_color=[0.0, 0.3],
    height=600,
    width=1000
)
fig.show()

In [ ]:
data[(data['month'] == 2)  & (data['depth_bin'] != 25.0)]

In [ ]:
fig = px.scatter_mapbox(
    data[(data['month'] == 8) & (data['depth_bin'] != 25.0)],
    lat='lat',
    lon='lon',
    color='min_risk_month',  # Color points by probability
    size='size',  # Adjust as needed
    size_max=11,  # Adjust as needed
    zoom=4,  # Adjust zoom level
    mapbox_style="carto-positron",  # Choose a map style,
    title='Spatial Pattern of Minimal Probability in Bottom Depth Bin in August',
    #range_color=[0.0, 0.3],
    height=600,
    width=1000
)
fig.show()

In [ ]:
sql = '''
select 
    *
from 
    chinook_depth_full_inference_3_1_18_2
where 
    month(time) in (2, 8) and day(time) = 15
    and elevation > -600
'''
data = read_data_w_cache(sql)
data['lat'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[0])
data['lon'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[1])
data['inside_polygon'] = data.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
data = data[data['inside_polygon']]
data = data[data['lon'] < -145]
print(data.shape)
data.head()

In [ ]:
data['max_depth_bin'] = data.groupby(['h3_index'])['depth_bin'].transform('max')
data['time'] = pd.to_datetime(data['epoch'], unit='s')
data['month'] = data['time'].dt.month
df = data[(data['depth_bin'] == data['max_depth_bin']) & (data['depth_bin'] != 25) & (data['depth_bin'] < 200)].groupby(['h3_index', 'month', 'depth_bin']).agg({
    'salinity': 'mean', 'mixed_layer_thickness': 'mean', 'nitrate': 'mean', 'probability': 'min'}).reset_index()
px.scatter(
    df, x='salinity', y='probability', facet_col='depth_bin',
    category_orders={'depth_bin': sorted(df['depth_bin'].unique())},
    facet_row='month', title='Salinity vs Probability in Bottom Depth Bin',
)

In [ ]:
px.scatter(
    df, x='mixed_layer_thickness', y='probability', facet_col='depth_bin',
    category_orders={'depth_bin': sorted(df['depth_bin'].unique())},
    facet_row='month', title='Mixed Layer Thickness vs Probability in Bottom Depth Bin'
)

In [ ]:
px.scatter(
    df, x='nitrate', y='probability', facet_col='depth_bin',
    category_orders={'depth_bin': sorted(df['depth_bin'].unique())},
    facet_row='month', title='Nitrate vs Probability in Bottom Depth Bin'
)

In [ ]:
h3_indices = {
    'Chignik': {
        'Coastal': '840ccebffffffff',
    }
}

sql = '''
select 
    epoch, h3_index, depth_bin, probability
from 
    chinook_depth_full_inference_3_1_18_2
where 
    h3_index = '{h3_index}'
'''
dfs = []
for place, cases in h3_indices.items():
    for case, h3_index in cases.items():
        data = read_data_w_cache(sql.format(h3_index=h3_index))
        data['time'] = pd.to_datetime(data['epoch'], unit='s', utc=False)
        # convert to alaska time
        data['time'] = data['time'].dt.tz_localize('UTC').dt.tz_convert('America/Anchorage')
        data['place'] = place
        data['case'] = case
        print(data.shape)
        dfs.append(data)
data = pd.concat(dfs).reset_index(drop=True)
print(data.shape)
data.head()

In [ ]:
h3.h3_to_geo('840ccebffffffff')

In [ ]:
color_discrete_map = {
    25.0: "#c7e9c0",  # Light Green
    50.0: "#a1d99b",  
    75.0: "#74c476",  
    100.0: "#41ab5d",  
    150.0: "#238b45",  
    200.0: "#1b7837",  
    250.0: "#0868ac",  
    300.0: "#08519c",  
    400.0: "#08306b",  # Deep Blue
    500.0: "#041c40",  # Darkest Blue (Deepest)
}

depth_order = sorted(color_discrete_map.keys())
data['depth_bin'] = pd.Categorical(data['depth_bin'], categories=depth_order, ordered=True)

# Now plot with the correct legend order
fig = px.line(
    data.sort_values('time'), x='time', y='probability', color='depth_bin',
    color_discrete_map=color_discrete_map,
    category_orders={"depth_bin": sorted(color_discrete_map.keys())},
    title='Likelihood per Depth Bin near Chignik'
)

fig.show()


In [ ]:
sql = '''
with risk as (
    select 
        month(time) as month, 
        epoch,
        h3_index,
        depth_bin,
        sin_sun,
        cos_sun,
        max(elevation) as elevation,
        sum(probability) as risk
    from 
        chinook_depth_full_inference_3_1_18_2
    where 
        day(time) = 15
    group by 
        1, 2, 3, 4, 5, 6
), max_depth_bin as (
    select 
        month(time) as month, 
        epoch,
        h3_index,
        max(depth_bin) as depth_bin
    from 
        chinook_depth_full_inference_3_1_18_2
    group by 
        1, 2, 3
), boundaries as (
    select 
        month,
        h3_index,
        depth_bin,
        elevation,
        approx_percentile(risk, 0.05) as min_risk_month,
        approx_percentile(risk, 0.95) as max_risk_month
    from 
        risk inner join max_depth_bin using (month, epoch, h3_index, depth_bin)
    group by 
        1, 2, 3, 4
), joined as (
    select 
        r.month,
        r.h3_index,
        r.sin_sun, 
        r.cos_sun,
        r.risk,
        b.depth_bin,
        b.elevation,
        b.min_risk_month,
        b.max_risk_month
    from 
        risk r
        inner join boundaries b 
            on r.month = b.month
            and r.h3_index = b.h3_index
)
select 
    month,
    h3_index,
    depth_bin,
    elevation,
    avg(sin_sun) as avg_sin_sun,
    avg(cos_sun) as avg_cos_sun
from 
    joined 
where 
    risk <= min_risk_month
group by 
    1, 2, 3, 4
'''
data = read_data_w_cache(sql)
data['lat'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[0])
data['lon'] = data['h3_index'].apply(lambda x: h3.h3_to_geo(x)[1])
print(data.shape)
data.head()

In [ ]:
data = data[data['depth_bin'] + 100 > -data['elevation']]
data['inside_polygon'] = data.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
data = data[data['inside_polygon']]
data = data[data['lon'] < -145]
data['size'] = 0.01

In [ ]:
data['sun_is_up'] = np.sign(data['avg_sin_sun'])
fig = px.scatter_mapbox(
    data[(data['month'] == 2) & (data['depth_bin'] != 25.0)],
    lat='lat',
    lon='lon',
    color='avg_sin_sun',  # Color points by probability
    size='size',  # Adjust as needed
    size_max=11,  # Adjust as needed
    zoom=4,  # Adjust zoom level
    mapbox_style="carto-positron",  # Choose a map style,
    title='Spatial Pattern of Time of Minimal Risk in February',
    range_color=[-1, 1],
    height=600,
    width=1000,
    color_continuous_scale='RdBu_r',  # Red to Blue color scale
)
fig.show()

In [ ]:
data['sun_is_up'] = np.sign(data['avg_sin_sun'])
fig = px.scatter_mapbox(
    data[(data['month'] == 8) & (data['depth_bin'] != 25.0)],
    lat='lat',
    lon='lon',
    color='avg_sin_sun',  # Color points by probability
    size='size',  # Adjust as needed
    size_max=11,  # Adjust as needed
    zoom=4,  # Adjust zoom level
    mapbox_style="carto-positron",  # Choose a map style,
    title='Spatial Pattern of Time of Minimal Risk in August',
    range_color=[-1, 1],
    height=600,
    width=1000,
    color_continuous_scale='RdBu_r',  # Red to Blue color scale
)
fig.show()

In [ ]:
print(df)
df = df[(df['depth_bin'] == 50) & (df['max_depth_bin'] == 50)]
column = 'mixed_layer_thickness'
df['bin'] = pd.qcut(df[column], q=3)
means = df.groupby('bin')[['min_probability', column]].mean().reset_index()
std = df.groupby('bin')[['min_probability']].std().reset_index()
means['std'] = std['min_probability']
px.line(
    means, x=column, y='min_probability', error_y='std'
)

In [ ]:
data['max_depth_bin'] = data.groupby(['h3_index'])['depth_bin'].transform('max')
data['min_probability'] = data.groupby(['h3_index'])['probability'].transform('min')
data['time'] = pd.to_datetime(data['epoch'], unit='s')
data['month'] = data['time'].dt.month
df = data[(data['month'] == 2)].groupby(['h3_index', 'depth_bin'])[[
    'min_probability', 'salinity', 'mixed_layer_thickness', 'nitrate', 'max_depth_bin', 
]].mean().reset_index()
df = df[(df['depth_bin'] == df['max_depth_bin']) & (df['max_depth_bin'] > 25) & (df['max_depth_bin'] < 250)]
df['min_probability_bin'] = df.groupby(['max_depth_bin'])['min_probability'].transform('max')
#df['min_probability'] = df['min_probability'] / df['min_probability_bin']

color_discrete_map = {
    25.0: "#c7e9c0",  # Light Green
    50.0: "#a1d99b",  
    75.0: "#74c476",  
    100.0: "#41ab5d",  
    150.0: "#238b45",  
    200.0: "#1b7837",  
    250.0: "#0868ac",  
    300.0: "#08519c",  
    400.0: "#08306b",  # Deep Blue
    500.0: "#041c40",  # Darkest Blue (Deepest)
}
depth_order = sorted(color_discrete_map.keys())
df['max_depth_bin'] = pd.Categorical(df['max_depth_bin'], categories=depth_order, ordered=True)




px.scatter(
    df, x='mixed_layer_thickness', y='min_probability', facet_col='max_depth_bin',
    height=500, category_orders={"max_depth_bin": sorted([k for k in color_discrete_map.keys() if k > 25 and k < 250])},
)

In [ ]:
column = 'salinity'
df = data[(data['month'] == 2) & (data['depth_bin'] == data['max_depth_bin']) & (data['depth_bin'] == 150)].groupby(['h3_index'])[['probability', column]].min().reset_index()

px.scatter(
    df, x=column, y='probability'
)

In [ ]:
df = data[(data['month'] == 2) & (data['depth_bin'] == data['max_depth_bin']) & (data['depth_bin'] == 75)].groupby(['h3_index'])[['probability', 'salinity', 'mixed_layer_thickness']].min().reset_index()
px.scatter(
    df, x='salinity', y='mixed_layer_thickness', color='probability'
)

In [ ]:
df[['salinity', 'probability', 'mixed_layer_thickness']].corr()

In [ ]:
data[['nitrate', 'salinity', 'mixed_layer_thickness']].corr()

In [ ]:
data['max_depth_bin'] = data.groupby(['h3_index'])['depth_bin'].transform('max')
df = data[data['max_depth_bin'] == 50]
df['time'] = pd.to_datetime(df['epoch'], unit='s')
df['month'] = df['time'].dt.month
df[(df['depth_bin'] == 25) & (df['month'] == 2)].groupby(['h3_index'])[['n_salinity', 'n_mixed_layer_thickness']].mean().reset_index()

In [ ]:
cases = {
    'low_salinity': '840c53dffffffff',
    'high_salinity': '840cce5ffffffff',
}
df = data[data['h3_index'].isin(cases.values())]
df['case'] = df['h3_index'].apply(lambda x: 'Low Salinity' if x == cases['low_salinity'] else 'High Salinity')
df.sort_values(by=['case', 'epoch'], inplace=True)
df['time'] = pd.to_datetime(df['epoch'], unit='s')
df['month'] = df['time'].dt.month
fig = px.line(
    df, x='time', y='probability', color='case', facet_col='depth_bin', facet_row='month',
)
for axis in fig.layout:
    if axis.startswith('xaxis'):
        fig.layout[axis]['matches'] = None
        
fig.show()

In [ ]:
px.line(
    df[df['month'] == 8], x='time', y='probability', color='case', facet_col='depth_bin', facet_row='month',
)

## Salinity

In [ ]:
sql = '''
select 
    _train,
    extract(month from time) as month,
    avg(ln(probability)) as loss
from
    chinook_depth_inference_3_7_2
where 
    run_id = 'fb3f06dc5fd0971a4e7cfdd2e5da5cca391a633f92528e79aa526df347ca0920'
    and _selected
group by 1, 2
order by 1
'''
salinity = read_data_w_cache(sql)
print(salinity.shape)
salinity.head()

In [ ]:
sql = '''
select 
    _train,
    extract(month from time) as month,
    avg(ln(probability)) as loss
from
    chinook_depth_inference_3_1_4
where 
    run_id = '00cf23b296999368ea18b82e33b8687c51e8c35e876afd325e26317cb69ea45b'
    and _selected
group by 1, 2
order by 1
'''
base = read_data_w_cache(sql)
base.head()

In [ ]:
df = base.merge(salinity, on=['month', '_train'], suffixes=('', '_w_salinity'))
df['difference'] = df['loss_w_salinity'] - df['loss']
df.sort_values(['month', '_train']).reset_index(drop=True)

In [ ]:
px.bar(df, x='month', y='difference', color='_train', barmode='group')

In [ ]:
sql = '''
with surface_salinity as (
    select
        _individual,
        _decision,
        extract(month from time) as month,
        salinity,
        probability,
        _train,
        h3_index
    from 
        chinook_depth_full_inference_3_7_2
    where 
        run_id = 'fb3f06dc5fd0971a4e7cfdd2e5da5cca391a633f92528e79aa526df347ca0920'
        and depth_bin = 25
), max_depth_bin as (
    select
        _individual,
        _decision,
        max(depth_bin) as max_depth_bin
    from 
        chinook_depth_full_inference_3_7_2
    where 
        run_id = 'fb3f06dc5fd0971a4e7cfdd2e5da5cca391a633f92528e79aa526df347ca0920'
    group by 
        1, 2
)
select
    ss.*,
    m.max_depth_bin
from 
    surface_salinity as ss
    inner join max_depth_bin m
        on ss._individual = m._individual
        and ss._decision = m._decision
'''
data = read_data_w_cache(sql)
print(data.shape)
data.head()

In [ ]:
boundary = data['salinity'].quantile(0.5)
data['salinity_bin'] = data['salinity'].map(lambda s: 'low' if s < boundary else 'high')

In [ ]:
gdf = data.groupby(
    ['month', 'max_depth_bin', '_train', 'salinity_bin']
).agg(
    {'probability': 'mean', 'salinity': 'mean', '_decision': 'nunique'}
).reset_index().rename(columns={'_decision': 'count'})
print(gdf.shape)
gdf.head()

In [ ]:
hdf = gdf[gdf['salinity_bin'] == 'high']
ldf = gdf[gdf['salinity_bin'] == 'low']
df = hdf.merge(ldf, on=['month', 'max_depth_bin'], suffixes=('_high', '_low'))
df['low - high'] = df['probability_low'] - df['probability_high']
px.bar(df.groupby(['month'])[['low - high']].mean().reset_index(), x='month', y='low - high')